# Machine Learning Assignment 2 — Model Implementation

**Dataset:** Breast Cancer Wisconsin (Diagnostic), UCI  
**Task:** Binary classification (`1 = malignant`, `0 = benign`)  
**Models:** Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, Random Forest, plus SVM as an additional sixth model.

> This notebook is designed to be executed on the BITS Virtual Lab for the required execution screenshot.


In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "breast_cancer_wisconsin_diagnostic.csv"
MODEL_DIR = BASE_DIR / "model"
RESULTS_DIR = BASE_DIR / "results"
TARGET = "diagnosis"
RANDOM_STATE = 42

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Environment ready.")


Environment ready.


## 1. Load and inspect the dataset

In [2]:
df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
print("\nTarget distribution:")
print(df[TARGET].value_counts().rename(index={0: "Benign", 1: "Malignant"}))
display(df.head())


Shape: (569, 31)

Target distribution:
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


## 2. Train/test split

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

test_data = X_test.copy()
test_data[TARGET] = y_test.values
test_data.to_csv(BASE_DIR / "test_data.csv", index=False)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Saved: test_data.csv")


Training shape: (455, 30)
Test shape: (114, 30)
Saved: test_data.csv


## 3. Define the classification models

In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=4, random_state=RANDOM_STATE
    ),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB())
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Support Vector Machine (Additional 6th)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ]),
}

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "K-Nearest Neighbors": "knn.joblib",
    "Gaussian Naive Bayes": "gaussian_naive_bayes.joblib",
    "Random Forest": "random_forest.joblib",
    "Support Vector Machine (Additional 6th)": "svm.joblib",
}

print("Models:", *models.keys(), sep="\n- ")


Models:
- Logistic Regression
- Decision Tree
- K-Nearest Neighbors
- Gaussian Naive Bayes
- Random Forest
- Support Vector Machine (Additional 6th)


## 4. Train, evaluate and save every model

In [5]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }
    results.append(metrics)
    trained_models[name] = model
    joblib.dump(model, MODEL_DIR / model_files[name])

metrics_df = pd.DataFrame(results)
metrics_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

display(metrics_df.round(4))


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.8772,0.9654,0.9118,0.7381,0.8158,0.7343
2,K-Nearest Neighbors,0.9561,0.9825,0.9744,0.9048,0.9383,0.9058
3,Gaussian Naive Bayes,0.9211,0.9891,0.9231,0.8571,0.8889,0.8292
4,Random Forest,0.9737,0.9954,1.0000,0.9286,0.9630,0.9442
5,Support Vector Machine (Additional 6th),0.9737,0.9947,1.0000,0.9286,0.9630,0.9442


## 5. Select an overall winner using all required metrics

In [6]:
metric_cols = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
ranked = metrics_df.copy()

for col in metric_cols:
    ranked[col + "_rank"] = ranked[col].rank(ascending=False, method="min")

ranked["Average Rank"] = ranked[[c + "_rank" for c in metric_cols]].mean(axis=1)
winner_row = ranked.sort_values(
    ["Average Rank", "AUC", "MCC"],
    ascending=[True, False, False]
).iloc[0]

print("Overall Winner:", winner_row["ML Model Name"])
print("Average metric rank:", round(winner_row["Average Rank"], 3))


Overall Winner: Random Forest
Average metric rank: 1.167


## 6. Detailed evaluation of the overall winner

In [7]:
winner_name = winner_row["ML Model Name"]
winner_model = trained_models[winner_name]
winner_pred = winner_model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, winner_pred))
print("\nClassification Report:")
print(classification_report(
    y_test, winner_pred, target_names=["Benign", "Malignant"], zero_division=0
))


Confusion Matrix:
[[72  0]
 [ 3 39]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



## 7. Files produced

- `test_data.csv`
- Six saved model files in `model/`
- `results/model_metrics.csv`

Next, run `streamlit run app.py` locally, push the complete folder to GitHub, deploy `app.py` on Streamlit Community Cloud, and execute this notebook in the BITS Virtual Lab for the required screenshot.
